# Phase I — Feature Engineering
## Goal: Convert cleaned text and categorical data into numeric features for ML

Techniques used:
- TF-IDF Vectorization → name_clean, desc_clean
- Label Encoding → cat1, cat2, cat3, brand_name
- Numeric features → item_condition_id, shipping, name_len, desc_len

Output: Feature matrix X and target vector y, saved for Phase II modeling

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack, csr_matrix
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded ✅")

Libraries loaded ✅


In [2]:
df = pd.read_pickle('../data/processed/train_cleaned.pkl')

print(f"Shape         : {df.shape}")
print(f"Columns       : {list(df.columns)}")
print(f"Missing values: {df.isnull().sum().sum()}")

Shape         : (1481661, 14)
Columns       : ['train_id', 'name', 'item_condition_id', 'category_name', 'brand_name', 'price', 'shipping', 'item_description', 'log_price', 'cat1', 'cat2', 'cat3', 'name_clean', 'desc_clean']
Missing values: 0


In [4]:
# Numeric Features + New Length Features

# name এবং description এ কতটা word আছে গুনছি
df['name_len'] = df['name_clean'].str.split().str.len()
df['desc_len'] = df['desc_clean'].str.split().str.len()

print("name_len stats:")
print(df['name_len'].describe().round(2))

print("\ndesc_len stats:")
print(df['desc_len'].describe().round(2))

print("\nSample:")
print(df[['name_clean', 'name_len', 'desc_clean', 'desc_len']].head(3).to_string())

name_len stats:
count    1481661.00
mean           4.40
std            1.65
min            0.00
25%            3.00
50%            4.00
75%            6.00
max           17.00
Name: name_len, dtype: float64

desc_len stats:
count    1481661.00
mean          25.58
std           30.23
min            0.00
25%            7.00
50%           15.00
75%           31.00
max          245.00
Name: desc_len, dtype: float64

Sample:
                            name_clean  name_len                                                                                                                                                                                    desc_clean  desc_len
0  mlb cincinnati reds t shirt size xl         7                                                                                                                                                                            no description yet         3
1     razer blackwidow chroma keyboard         4  this keyboard is in great co

In [6]:
# Label Encoding

# for 4 column 4 distinct LabelEncoder
le_cat1  = LabelEncoder()
le_cat2  = LabelEncoder()
le_cat3  = LabelEncoder()
le_brand = LabelEncoder()

df['cat1_encoded']  = le_cat1.fit_transform(df['cat1'])
df['cat2_encoded']  = le_cat2.fit_transform(df['cat2'])
df['cat3_encoded']  = le_cat3.fit_transform(df['cat3'])
df['brand_encoded'] = le_brand.fit_transform(df['brand_name'])

print("Label Encoding complete ✅")
print(f"\nUnique values encoded:")
print(f"  cat1  : {df['cat1_encoded'].nunique()} categories")
print(f"  cat2  : {df['cat2_encoded'].nunique()} categories")
print(f"  cat3  : {df['cat3_encoded'].nunique()} categories")
print(f"  brand : {df['brand_encoded'].nunique()} brands")

print("\nSample — cat1 before and after:")
sample = df[['cat1', 'cat1_encoded']].drop_duplicates().head(8)
print(sample.to_string())

Label Encoding complete ✅

Unique values encoded:
  cat1  : 11 categories
  cat2  : 114 categories
  cat3  : 872 categories
  brand : 4808 brands

Sample — cat1 before and after:
                      cat1  cat1_encoded
0                      Men             5
1              Electronics             1
2                    Women             9
3                     Home             3
7        Sports & Outdoors             7
9   Vintage & Collectibles             8
10                  Beauty             0
13                   Other             6


In [8]:
# TF-IDF on name_clean

print("Fitting TF-IDF on product names...")


tfidf_name = TfidfVectorizer(
    max_features=50000,   # keep highest 50k words
    ngram_range=(1, 2),   # single words + word pairs
    strip_accents='unicode',
    analyzer='word',
    token_pattern=r'\w{2,}',  # minimum 2 character words
    min_df=3              #  keep it if has at least 3 listing
)

X_name = tfidf_name.fit_transform(df['name_clean'])

print(f"\nTF-IDF (name) matrix shape : {X_name.shape}")
print(f"Type                       : {type(X_name)}")
print(f"Non-zero elements          : {X_name.nnz:,}")
print(f"Sparsity                   : {(1 - X_name.nnz / (X_name.shape[0] * X_name.shape[1])):.3%}")


Fitting TF-IDF on product names...

TF-IDF (name) matrix shape : (1481661, 50000)
Type                       : <class 'scipy.sparse._csr.csr_matrix'>
Non-zero elements          : 8,772,013
Sparsity                   : 99.988%


In [9]:
#  TF-IDF on desc_clean

print("Fitting TF-IDF on item descriptions...")


tfidf_desc = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    strip_accents='unicode',
    analyzer='word',
    token_pattern=r'\w{2,}',
    min_df=3
)

X_desc = tfidf_desc.fit_transform(df['desc_clean'])

print(f"\nTF-IDF (desc) matrix shape : {X_desc.shape}")
print(f"Non-zero elements          : {X_desc.nnz:,}")
print(f"Sparsity                   : {(1 - X_desc.nnz / (X_desc.shape[0] * X_desc.shape[1])):.3%}")

Fitting TF-IDF on item descriptions...

TF-IDF (desc) matrix shape : (1481661, 50000)
Non-zero elements          : 49,225,868
Sparsity                   : 99.934%


In [10]:
# Combine All Features

# Numeric features in one matrix
numeric_cols = [
    'item_condition_id',
    'shipping',
    'name_len',
    'desc_len',
    'cat1_encoded',
    'cat2_encoded',
    'cat3_encoded',
    'brand_encoded'
]

X_numeric = csr_matrix(df[numeric_cols].values)

print(f"Numeric features shape : {X_numeric.shape}")

# concat all features

X = hstack([X_name, X_desc, X_numeric])

print(f"\nFinal feature matrix shape : {X.shape}")
print(f"Total features             : {X.shape[1]:,}")

# Target vector
y = df['log_price'].values
print(f"\nTarget vector shape        : {y.shape}")
print(f"Target range               : {y.min():.3f} to {y.max():.3f}")

Numeric features shape : (1481661, 8)

Final feature matrix shape : (1481661, 100008)
Total features             : 100,008

Target vector shape        : (1481661,)
Target range               : 1.386 to 7.606


In [11]:
# Save everything

os.makedirs('../data/processed', exist_ok=True)

# Save Feature matrix and target
joblib.dump(X, '../data/processed/X_features.pkl')
joblib.dump(y, '../data/processed/y_target.pkl')

# Encoders save  — need in Phase III deployment 
joblib.dump(tfidf_name,  '../data/processed/tfidf_name.pkl')
joblib.dump(tfidf_desc,  '../data/processed/tfidf_desc.pkl')
joblib.dump(le_cat1,     '../data/processed/le_cat1.pkl')
joblib.dump(le_cat2,     '../data/processed/le_cat2.pkl')
joblib.dump(le_cat3,     '../data/processed/le_cat3.pkl')
joblib.dump(le_brand,    '../data/processed/le_brand.pkl')

print("Saved files:")
for f in os.listdir('../data/processed'):
    size = os.path.getsize(f'../data/processed/{f}') / 1e6
    print(f"  {f:<35} {size:.1f} MB")

Saved files:
  tfidf_desc.pkl                      2.0 MB
  le_brand.pkl                        0.1 MB
  tfidf_name.pkl                      2.0 MB
  train_cleaned.pkl                   652.5 MB
  price_vs_features.png               0.0 MB
  le_cat1.pkl                         0.0 MB
  le_cat2.pkl                         0.0 MB
  X_features.pkl                      831.7 MB
  le_cat3.pkl                         0.0 MB
  y_target.pkl                        11.9 MB
  price_distribution.png              0.0 MB


## Feature Engineering Complete ✅

| Feature Group | Technique | Columns | Output Shape |
|---|---|---|---|
| Product name | TF-IDF (50k, bigram) | name_clean | (1.48M × 50,000) |
| Description | TF-IDF (50k, bigram) | desc_clean | (1.48M × 50,000) |
| Categories | Label Encoding | cat1, cat2, cat3 | (1.48M × 3) |
| Brand | Label Encoding | brand_name | (1.48M × 1) |
| Numeric | Direct use | condition, shipping, name_len, desc_len | (1.48M × 4) |
| **FINAL** | **hstack** | **All combined** | **(1.48M × ~100,006)** |

All encoders saved to data/processed/ for Phase III deployment.